# 📊 Exploratory Data Analysis (EDA) - Telco Customer Churn

This notebook serves as the initial data investigation workspace to understand subscriber behaviors, identify data cleaning needs, and unearth feature relationships before running the machine learning pipeline.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting aesthetic profiles
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

print("✅ Setup complete. Analysis libraries initialized.")

## 1. Load Data & Schema Overview

In [ ]:
# Load locally saved baseline raw file
df = pd.read_csv("../data/telco_raw.csv")
print(f"Dataset Schema Dimensions: {df.shape[0]} Rows, {df.shape[1]} Columns\n")
df.info()

## 2. Identify Missing Values & Hidden Spaces
Notice that `TotalCharges` is interpreted as an `object` string type because it contains blank space entries (`' '`). We must convert these to numeric values and drop the resulting nulls.

In [ ]:
# Count hidden empty string rows
hidden_missing = (df['TotalCharges'] == ' ').sum()
print(f"⚠️ Found {hidden_missing} rows with blank string values in TotalCharges.")

# Coerce to numeric data types and drop empty cells
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(subset=['TotalCharges'], inplace=True)
print(f"✅ Cleaned Dataset Dimensions: {df.shape[0]} Rows remaining.")

## 3. Analyze Target Class Balance
Let's check the distribution of the target variable `Churn` to evaluate class imbalance severity.

In [ ]:
churn_counts = df['Churn'].value_counts(normalize=True) * 100
print("=== TARGET DISTRIBUTION ===")
print(f"Loyal (No Churn): {churn_counts['No']:.2f}%")
print(f"Cancelled (Churn): {churn_counts['Yes']:.2f}%")

fig, ax = plt.subplots(figsize=(6, 4))
sns.countplot(x='Churn', data=df, palette='Blues_r', ax=ax)
ax.set_title("Distribution of Customer Cancellation Target Status")
plt.show()

## 4. Explore Numerical Relationships
We investigate how numerical variables scale against customer cancellation frequencies. Younger accounts (low tenure) with higher monthly service bills display extreme risk signals.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Tenure distribution mapping
sns.kdeplot(data=df, x='tenure', hue='Churn', shade=True, common_norm=False, palette='Set1', ax=ax1)
ax1.set_title("Account Tenure Distribution Shift by Churn")

# Monthly charges distribution mapping
sns.kdeplot(data=df, x='MonthlyCharges', hue='Churn', shade=True, common_norm=False, palette='Set1', ax=ax2)
ax2.set_title("Monthly Bill Scale Distribution Shift by Churn")

plt.tight_layout()
plt.show()

## 5. Map Feature Correlations
Let's look at a correlation heatmap of our continuous numerical values to spot collinearity.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
numerical_corr = df[['tenure', 'MonthlyCharges', 'TotalCharges']].corr()
sns.heatmap(numerical_corr, annot=True, cmap='coolwarm', fmt=".2f", linewidths=0.5, ax=ax)
ax.set_title("Numerical Continuous Values Correlation Matrix")
plt.show()